Tutorial 3 (CH-4124)

####
Fitting chemical data
From raw numbers to a physical parameter: least squares, nonlinear fits, and log-linearisation.
####

### 1. NumPy warm-up and vectorised chemistry

Build arrays, apply the Beer-Lambert law elementwise, and add measurement noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# Concentration grid (mol/L)
c = ...          # TO DO: np.linspace(0.0, 1.0e-3, 25)

epsilon = ...    # TO DO: molar absorptivity, e.g., 8500.0 (L/mol/cm)
path_len = ...   # TO DO: cuvette path length in cm, e.g., 1.0

# Beer-Lambert: A = epsilon * l * c
A_true = ...     # TO DO: vectorised expression, no loop

# Add gaussian noise to mimic a real spectrometer
noise = rng.normal(loc=0.0, scale=..., size=c.size)   # TO DO: scale e.g. 0.02
A_obs = ...      # TO DO: A_true + noise

print("shape:", A_obs.shape, "| max A:", A_obs.max().round(3))
print("mean:", ..., "| std:", ...)   # TO DO: A_obs.mean(), A_obs.std()

### 2. Straight-line least squares

Fit A vs c, recover the molar absorptivity, and judge the fit quality with R-squared.

In [ ]:
# Fit a first degree polynomial: A = m*c + b
coeffs = ...        # TO DO: np.polyfit(c, A_obs, 1)
m, b = ...          # TO DO: unpack coeffs

A_fit = ...         # TO DO: np.polyval(coeffs, c)

# R-squared
ss_res = ...        # TO DO: np.sum((A_obs - A_fit)**2)
ss_tot = ...        # TO DO: np.sum((A_obs - A_obs.mean())**2)
r2 = ...            # TO DO: 1 - ss_res/ss_tot

print("slope (eps*l):", m)
print("recovered epsilon:", ...)     # TO DO: m / path_len
print("intercept:", b, "| R2:", r2)

plt.figure(figsize=(6, 4))
plt.scatter(c, A_obs, s=..., alpha=..., label="data")     # TO DO: e.g. 30, 0.8
plt.plot(c, A_fit, "-", label=f"fit, R2 = {r2:.3f}")
plt.xlabel("concentration, mol/L")
plt.ylabel("absorbance")
plt.title("Beer-Lambert calibration")
plt.legend()
plt.grid(True)

### 3. Residuals

A fit is only as good as its residuals. Plot them and look for structure.

In [ ]:
resid = ...      # TO DO: A_obs - A_fit

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(..., ..., s=25)        # TO DO: c, resid
axes[0].axhline(0.0, ls="--", lw=1)
axes[0].set_xlabel("concentration, mol/L")
axes[0].set_ylabel("residual")
axes[0].set_title("Residuals vs x")
axes[0].grid(True)

axes[1].hist(..., bins=...)            # TO DO: resid, e.g. 10
axes[1].set_xlabel("residual")
axes[1].set_ylabel("count")
axes[1].set_title("Residual distribution")
axes[1].grid(True)

plt.tight_layout()

print("mean residual:", ...)           # TO DO: resid.mean()
print("RMSE:", ...)                    # TO DO: np.sqrt(np.mean(resid**2))

### 4. Nonlinear fit — first order kinetics

A decays as [A] = [A]0 * exp(-k t). Fit it directly with `curve_fit`, then compare
against the linearised form ln[A] vs t.

In [ ]:
from scipy.optimize import curve_fit

# Synthetic kinetics run
t = np.linspace(0.0, 60.0, 30)          # minutes
A0_true, k_true = 0.85, 0.045
conc = A0_true*np.exp(-k_true*t) + rng.normal(0.0, 0.015, t.size)

def first_order(t, A0, k):
    return ...                          # TO DO: A0*np.exp(-k*t)

p0 = ...                                # TO DO: initial guess, e.g. [1.0, 0.01]
popt, pcov = ...                        # TO DO: curve_fit(first_order, t, conc, p0=p0)

A0_fit, k_fit = ...                     # TO DO: unpack popt
perr = ...                              # TO DO: np.sqrt(np.diag(pcov))

print(f"A0 = {A0_fit:.4f} +/- {perr[0]:.4f}")
print(f"k  = {k_fit:.5f} +/- {perr[1]:.5f} 1/min")
print("half life, min:", ...)           # TO DO: np.log(2)/k_fit

t_smooth = np.linspace(t.min(), t.max(), 200)

plt.figure(figsize=(6, 4))
plt.scatter(t, conc, s=30, alpha=0.8, label="data")
plt.plot(t_smooth, ..., "-", label="fit")   # TO DO: first_order(t_smooth, *popt)
plt.xlabel("time, min")
plt.ylabel("[A], mol/L")
plt.title("First order decay")
plt.legend()
plt.grid(True)

In [ ]:
# Linearised check: ln[A] should be straight with slope -k
mask = conc > 0                    # guard against log of a negative noisy point
ln_c = ...                         # TO DO: np.log(conc[mask])

slope, intercept = ...             # TO DO: np.polyfit(t[mask], ln_c, 1)
print("k from linearisation:", ...)   # TO DO: -slope
print("k from curve_fit    :", k_fit)

plt.figure(figsize=(6, 4))
plt.scatter(t[mask], ln_c, s=30, alpha=0.8)
plt.plot(t[mask], ..., "-")        # TO DO: np.polyval([slope, intercept], t[mask])
plt.xlabel("time, min")
plt.ylabel("ln [A]")
plt.title("Linearised first order plot")
plt.grid(True)

### 5. New dataset — Arrhenius rate constants

Work with `arrhenius_rates.csv`, columns `run_id`, `temperature_K`, `rate_constant`, `catalyst`.
Tasks:

Read the CSV and inspect it
Add columns for 1/T and ln k
Fit ln k vs 1/T for each catalyst separately
Report Ea in kJ/mol and the pre-exponential factor A
Overlay both fits on one plot

In [ ]:
import pandas as pd

path = "arrhenius_rates.csv"

df = ...                 # TO DO: pd.read_csv(path)
print(df.head())
print(df.info())
print(df["catalyst"].value_counts())

# Derived columns
df["inv_T"] = ...        # TO DO: 1.0 / df["temperature_K"]
df["ln_k"]  = ...        # TO DO: np.log(df["rate_constant"])

df.head()

In [ ]:
R = 8.314   # J/mol/K

results = {}

for cat, grp in ...:                       # TO DO: df.groupby("catalyst")
    x = grp["inv_T"].to_numpy()
    y = ...                                # TO DO: grp["ln_k"].to_numpy()

    slope, intercept = ...                 # TO DO: np.polyfit(x, y, 1)

    Ea_kJ = ...                            # TO DO: -slope * R / 1000.0
    A_pre = ...                            # TO DO: np.exp(intercept)

    results[cat] = {"slope": slope, "intercept": intercept,
                    "Ea_kJ_per_mol": Ea_kJ, "A": A_pre, "n": len(grp)}

    print(f"{cat:>12} | Ea = {Ea_kJ:7.2f} kJ/mol | A = {A_pre:.3e} | n = {len(grp)}")

summary = pd.DataFrame(results).T
summary

In [ ]:
plt.figure(figsize=(6.5, 4.5))

for cat, grp in df.groupby("catalyst"):
    x = grp["inv_T"].to_numpy()
    y = grp["ln_k"].to_numpy()
    plt.scatter(x, y, s=..., alpha=..., label=cat)          # TO DO: e.g. 35, 0.8

    fit = results[cat]
    xs = np.linspace(x.min(), x.max(), 50)
    plt.plot(xs, ..., "--")                                  # TO DO: fit["slope"]*xs + fit["intercept"]

plt.xlabel("1/T, 1/K")
plt.ylabel("ln k")
plt.title(...)                                               # TO DO: give it a title
plt.legend()
plt.grid(...)                                                # TO DO

### 6. Messy data

Real files have blanks and outliers. Clean before fitting.

In [ ]:
# How many missing values per column?
print(...)                        # TO DO: df.isna().sum()

# Drop rows missing the rate constant
df_clean = ...                    # TO DO: df.dropna(subset=["rate_constant"]).copy()
print("rows before:", len(df), "| after:", len(df_clean))

# Flag outliers with a z-score on ln k, per catalyst
z = df_clean.groupby("catalyst")["ln_k"].transform(lambda s: (s - s.mean()) / s.std())
df_clean["is_outlier"] = ...      # TO DO: z.abs() > 2.5
print(df_clean["is_outlier"].sum(), "outliers flagged")

df_ok = ...                       # TO DO: df_clean[~df_clean["is_outlier"]]

# Refit one catalyst with and without outliers and compare Ea
cat = ...                         # TO DO: pick a catalyst name from the file
sub_all = df_clean[df_clean["catalyst"] == cat]
sub_ok  = df_ok[df_ok["catalyst"] == cat]

for name, sub in [("with outliers", sub_all), ("cleaned", sub_ok)]:
    s, _ = np.polyfit(sub["inv_T"], sub["ln_k"], 1)
    print(f"{name:>14}: Ea = {-s*R/1000:.2f} kJ/mol")

### 7. Publication-style figure

Two panels side by side, shared styling, saved to disk at 300 dpi.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(..., ...))   # TO DO: e.g. 10, 4

# Left: k vs T
for cat, grp in df.groupby("catalyst"):
    ax1.plot(grp["temperature_K"], grp["rate_constant"], "o", ms=5, label=cat)
ax1.set_yscale(...)                # TO DO: "log"
ax1.set_xlabel("T, K")
ax1.set_ylabel("k")
ax1.legend(frameon=False)
ax1.grid(True, alpha=0.3)

# Right: Arrhenius
for cat, grp in df.groupby("catalyst"):
    ax2.plot(..., ..., "o", ms=5, label=cat)    # TO DO: grp["inv_T"], grp["ln_k"]
ax2.set_xlabel("1/T, 1/K")
ax2.set_ylabel("ln k")
ax2.grid(True, alpha=0.3)

fig.suptitle("Temperature dependence of the rate constant")
fig.tight_layout()
fig.savefig(..., dpi=...)          # TO DO: "arrhenius_figure.png", 300

### 8. Practice on your own

Repeat the whole pipeline on `enzyme_kinetics.csv` with columns
`substrate_mM`, `velocity_uM_per_min`, `enzyme_batch`.

1. Fit the Michaelis-Menten form v = Vmax*S / (Km + S) with `curve_fit`
2. Report Vmax and Km with their uncertainties for each batch
3. Make a Lineweaver-Burk plot (1/v vs 1/S) and fit it linearly
4. Compare Km from the nonlinear fit against Km from the linearised fit,
   and write one or two sentences on why they disagree

In [ ]:
def michaelis_menten(S, Vmax, Km):
    return ...        # TO DO

# TO DO: read the file, loop over enzyme_batch, fit, print, plot